In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if that is your actual folder name.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_J.xpt"
csv_file = DATA_DIR / "DIQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2017-2018 DIQ_J XPT -> CSV")
print("=" * 100)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040",
    "DIQ160",
    "DIQ170",
    "DIQ172",
    "DIQ175A",
    "DIQ175B",
    "DIQ175C",
    "DIQ175D",
    "DIQ175E",
    "DIQ175F",
    "DIQ175G",
    "DIQ175H",
    "DIQ175I",
    "DIQ175J",
    "DIQ175K",
    "DIQ175L",
    "DIQ175M",
    "DIQ175N",
    "DIQ175O",
    "DIQ175P",
    "DIQ175Q",
    "DIQ175R",
    "DIQ175S",
    "DIQ175T",
    "DIQ175U",
    "DIQ175V",
    "DIQ175W",
    "DIQ175X",
    "DIQ180",
    "DIQ050",
    "DID060",
    "DIQ060U",
    "DIQ070",
    "DIQ230",
    "DIQ240",
    "DID250",
    "DID260",
    "DIQ260U",
    "DIQ275",
    "DIQ280",
    "DIQ291",
    "DIQ300S",
    "DIQ300D",
    "DID310S",
    "DID310D",
    "DID320",
    "DID330",
    "DID341",
    "DID350",
    "DIQ350U",
    "DIQ360",
    "DIQ080"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DIQ_J columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8897:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 54:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,897 rows x 54 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached DIQ_J:
#
#     Character fields       = NONE
#
#     Genuine decimal field  = DIQ280
#
#     Whole-number fields    = remaining 53
#
#
# DIQ280:
#
#     What was your last A1C level?
#
# It contains genuine decimal values such as:
#
#     5.4
#     6.7
#     7.2
#
# Therefore DIQ280 MUST remain float.
#
#
# IMPORTANT:
#
# We do not identify decimal fields merely from their
# column dtype.
#
# We inspect the actual released values.
#
# This same principle protects genuine Dietary,
# Laboratory, Examination, and weight decimals.
#

STRING_COLUMNS = []

DECIMAL_COLUMNS = [
    "DIQ280"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(INTEGER_COLUMNS) != 53:

    raise ValueError(
        f"Expected 53 integer/code fields, "
        f"found {len(INTEGER_COLUMNS)}."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# NEVER globally replace it before validating
# the exact variables and counts.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 54 DIQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{ordinary_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached DIQ_J.xpt:
#
# DID250       9
# DID260     178
# DID341     244
# DID350     168
#
# TOTAL      599
#
#
# CDC confirms:
#
# DID250 = 0
#     None
#     Count = 9
#
# DID260 = 0
#     Never
#     Count = 178
#
# DID341 = 0
#     None
#     Count = 244
#
# DID350 = 0
#     None
#     Count = 168
#

EXPECTED_TINY_COUNTS = {
    "DID250": 9,
    "DID260": 178,
    "DID341": 244,
    "DID350": 168
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DIQ_J attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 599:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 4:

    raise ValueError(
        "Expected tiny values in exactly 4 variables."
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before tiny-value correction."
    )


print(
    "\nPASS: Exact 599-value tiny-zero pattern "
    "across 4 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 599:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE REFERENCE
# ============================================================

#
# expected_df is our exact numerical reference.
#
# ONLY the validated tiny -> zero repairs are allowed.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after repair."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY diagnostically.
#
# NO dataframe values are overwritten.
#

print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    if count > 0:

        fractional_columns.append(
            col
        )


print(
    "Variables with fractional observations:",
    fractional_columns
)


if fractional_columns != ["DIQ280"]:

    raise ValueError(
        "Unexpected fractional-variable pattern.\n\n"
        "Expected:\n['DIQ280']\n\n"
        f"Actual:\n{fractional_columns}"
    )


if fractional_counts["DIQ280"] != 342:

    raise ValueError(
        "Unexpected DIQ280 fractional count."
    )


print(
    "PASS: DIQ280 is the only genuine "
    "decimal variable."
)

print(
    "DIQ280 fractional observations:",
    fractional_counts["DIQ280"]
)


# ============================================================
# 11. VALIDATE DIQ280 A1C WITHOUT CHANGING IT
# ============================================================

#
# CDC:
#
#     2.3-14 = 490
#     777    = 2
#     999    = 255
#     Missing = 8,150
#
#
# This is the critical decimal-preservation field.
#
# DO NOT:
#
#     round()
#     astype(Int64)
#     truncate
#

diq280_valid = int(
    df["DIQ280"]
    .between(
        2.3,
        14
    )
    .sum()
)


diq280_777 = int(
    df["DIQ280"]
    .eq(777)
    .sum()
)


diq280_999 = int(
    df["DIQ280"]
    .eq(999)
    .sum()
)


diq280_missing = int(
    df["DIQ280"]
    .isna()
    .sum()
)


diq280_fractional = int(
    fractional_counts["DIQ280"]
)


diq280_valid_values = (
    df.loc[
        df["DIQ280"]
        .between(
            2.3,
            14
        ),
        "DIQ280"
    ]
    .to_numpy(
        dtype=float
    )
)


diq280_valid_fractional = int(
    (
        ~np.isclose(
            diq280_valid_values,
            np.round(
                diq280_valid_values
            ),
            rtol=0,
            atol=1e-12
        )
    )
    .sum()
)


diq280_valid_whole = (
    diq280_valid
    -
    diq280_valid_fractional
)


print(
    "\n--- DIQ280 A1C FIDELITY ---"
)

print(
    "Valid 2.3-14:",
    diq280_valid
)

print(
    "Fractional valid A1C:",
    diq280_valid_fractional
)

print(
    "Whole-looking valid A1C:",
    diq280_valid_whole
)

print(
    "777:",
    diq280_777
)

print(
    "999:",
    diq280_999
)

print(
    "Missing:",
    f"{diq280_missing:,}"
)


if diq280_valid != 490:

    raise ValueError(
        "DIQ280 valid-range count differs."
    )


if diq280_valid_fractional != 342:

    raise ValueError(
        "DIQ280 genuine fractional count differs."
    )


if diq280_valid_whole != 148:

    raise ValueError(
        "DIQ280 whole-looking valid count differs."
    )


if diq280_777 != 2:

    raise ValueError(
        "DIQ280 refused count differs."
    )


if diq280_999 != 255:

    raise ValueError(
        "DIQ280 don't-know count differs."
    )


if diq280_missing != 8150:

    raise ValueError(
        "DIQ280 missing count differs."
    )


print(
    "PASS: DIQ280 A1C values validated "
    "WITHOUT modification."
)


# ============================================================
# 12. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 13. PROVE THE OTHER 53 VARIABLES ARE WHOLE-NUMBER
# ============================================================

integer_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        integer_failures.append(
            col
        )


if integer_failures:

    raise ValueError(
        "Unexpected fractional integer/code fields:\n"
        + str(
            integer_failures
        )
    )


print(
    "PASS: All 53 non-DIQ280 fields are "
    "mathematically whole-number."
)


# ============================================================
# 14. CONVERT ONLY PROVEN INTEGER FIELDS TO Int64
# ============================================================

#
# CRITICAL:
#
# DIQ280 is NOT included.
#
# NO .round() is used during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: 53 whole-number fields converted "
    "to nullable Int64."
)


if not pd.api.types.is_float_dtype(
    df["DIQ280"].dtype
):

    raise ValueError(
        "DIQ280 is no longer floating-point."
    )


print(
    "PASS: DIQ280 remains floating-point."
)


# ============================================================
# 15. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column}"
    )


# ============================================================
# 16. HELPER - RANGE + SPECIAL CODES
# ============================================================

def check_range_special(
    column,
    minimum,
    maximum,
    expected_range_count,
    expected_special,
    expected_missing
):

    range_count = int(
        df[column]
        .between(
            minimum,
            maximum
        )
        .sum()
    )


    special_actual = {
        code: int(
            df[column]
            .eq(code)
            .sum()
        )
        for code in expected_special
    }


    missing_count = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} RANGE CHECK ---"
    )

    print(
        f"{minimum}-{maximum}:",
        f"{range_count:,}"
    )

    print(
        "Special codes:",
        special_actual
    )

    print(
        "Missing:",
        f"{missing_count:,}"
    )


    if range_count != expected_range_count:

        raise ValueError(
            f"{column} valid-range count differs."
        )


    if special_actual != expected_special:

        raise ValueError(
            f"{column} special-code counts differ."
        )


    if missing_count != expected_missing:

        raise ValueError(
            f"{column} missing count differs."
        )


    print(
        f"PASS: {column}"
    )


# ============================================================
# 17. DIQ010
# ============================================================

check_frequency(
    "DIQ010",
    {
        1: 893,
        2: 7816,
        3: 184,
        9: 4
    },
    0
)


# ============================================================
# 18. DID040
# ============================================================

check_range_special(
    "DID040",
    1,
    79,
    870,
    {
        80: 11,
        666: 1,
        777: 0,
        999: 10
    },
    8005
)


# ============================================================
# 19. PREDIABETES / RISK QUESTIONS
# ============================================================

check_frequency(
    "DIQ160",
    {
        1: 597,
        2: 5089,
        9: 8
    },
    3203
)


check_frequency(
    "DIQ170",
    {
        1: 948,
        2: 4911,
        9: 17
    },
    3021
)


check_frequency(
    "DIQ172",
    {
        1: 1562,
        2: 4243,
        7: 2,
        9: 69
    },
    3021
)


# ============================================================
# 20. DIQ175A-X MULTI-RESPONSE RISK FACTORS
# ============================================================

EXPECTED_DIQ175 = {

    "DIQ175A": {
        10: 1143,
        99: 2
    },

    "DIQ175B": {
        11: 657
    },

    "DIQ175C": {
        12: 217
    },

    "DIQ175D": {
        13: 488
    },

    "DIQ175E": {
        14: 177
    },

    "DIQ175F": {
        15: 38
    },

    "DIQ175G": {
        16: 318
    },

    "DIQ175H": {
        17: 291
    },

    "DIQ175I": {
        18: 76
    },

    "DIQ175J": {
        19: 163
    },

    "DIQ175K": {
        20: 29
    },

    "DIQ175L": {
        21: 71
    },

    "DIQ175M": {
        22: 155
    },

    "DIQ175N": {
        23: 113
    },

    "DIQ175O": {
        24: 167
    },

    "DIQ175P": {
        25: 143
    },

    "DIQ175Q": {
        26: 172
    },

    "DIQ175R": {
        27: 19
    },

    "DIQ175S": {
        28: 37
    },

    "DIQ175T": {
        29: 116
    },

    "DIQ175U": {
        30: 113
    },

    "DIQ175V": {
        31: 14
    },

    "DIQ175W": {
        32: 5
    },

    "DIQ175X": {
        33: 2
    }
}


print(
    "\n--- DIQ175A-X RISK FACTORS ---"
)


for col, expected_freq in EXPECTED_DIQ175.items():

    actual_freq = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    print(
        f"{col}:",
        actual_freq
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"{col} frequency pattern differs."
        )


print(
    "PASS: DIQ175A-X frequencies validated."
)


# ============================================================
# 21. DIQ172 -> DIQ175 ROUTING
# ============================================================

risk_columns = [
    "DIQ175A",
    "DIQ175B",
    "DIQ175C",
    "DIQ175D",
    "DIQ175E",
    "DIQ175F",
    "DIQ175G",
    "DIQ175H",
    "DIQ175I",
    "DIQ175J",
    "DIQ175K",
    "DIQ175L",
    "DIQ175M",
    "DIQ175N",
    "DIQ175O",
    "DIQ175P",
    "DIQ175Q",
    "DIQ175R",
    "DIQ175S",
    "DIQ175T",
    "DIQ175U",
    "DIQ175V",
    "DIQ175W",
    "DIQ175X"
]


risk_expected = (
    df["DIQ172"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


risk_present = (
    df[risk_columns]
    .notna()
    .any(axis=1)
    .to_numpy(
        dtype=bool
    )
)


risk_routing_mismatch = int(
    np.sum(
        risk_expected
        !=
        risk_present
    )
)


print(
    "\n--- DIQ172 / DIQ175 ROUTING ---"
)

print(
    "DIQ172=1:",
    int(
        risk_expected.sum()
    )
)

print(
    "Any DIQ175 response:",
    int(
        risk_present.sum()
    )
)

print(
    "Routing mismatches:",
    risk_routing_mismatch
)


if int(
    risk_expected.sum()
) != 1562:

    raise ValueError(
        "Unexpected DIQ175 eligibility count."
    )


if risk_routing_mismatch != 0:

    raise ValueError(
        "DIQ172/DIQ175 routing differs."
    )


print(
    "PASS: DIQ172/DIQ175 routing validated."
)


# ============================================================
# 22. DIQ180
# ============================================================

check_frequency(
    "DIQ180",
    {
        1: 2679,
        2: 3016,
        7: 1,
        9: 180
    },
    3021
)


# ============================================================
# 23. DIQ050 - TAKING INSULIN
# ============================================================

check_frequency(
    "DIQ050",
    {
        1: 254,
        2: 8642,
        9: 1
    },
    0
)


# ============================================================
# 24. DID060 - INSULIN DURATION
# ============================================================

check_range_special(
    "DID060",
    1,
    59,
    245,
    {
        666: 7,
        777: 0,
        999: 2
    },
    8643
)


# ============================================================
# 25. DIQ050 -> DID060 ROUTING
# ============================================================

did060_expected = (
    df["DIQ050"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


did060_present = (
    df["DID060"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


did060_mismatch = int(
    np.sum(
        did060_expected
        !=
        did060_present
    )
)


if did060_mismatch != 0:

    raise ValueError(
        "DIQ050/DID060 routing differs."
    )


print(
    "PASS: DIQ050/DID060 routing validated."
)


# ============================================================
# 26. DIQ060U
# ============================================================

check_frequency(
    "DIQ060U",
    {
        1: 36,
        2: 209
    },
    8652
)


diq060u_expected = (
    df["DID060"]
    .between(
        1,
        59
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq060u_present = (
    df["DIQ060U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        diq060u_expected
        !=
        diq060u_present
    )
) != 0:

    raise ValueError(
        "DID060/DIQ060U routing differs."
    )


print(
    "PASS: DID060/DIQ060U routing validated."
)


# ============================================================
# 27. DIQ070 - DIABETIC PILLS
# ============================================================

check_frequency(
    "DIQ070",
    {
        1: 717,
        2: 955,
        9: 2
    },
    7223
)


#
# CDC eligibility:
#
# DIQ010 = 1
# OR
# DIQ010 = 3
# OR
# DIQ160 = 1
#

diq070_expected = (
    (
        df["DIQ010"].isin(
            [1, 3]
        )
        |
        df["DIQ160"].eq(1)
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq070_present = (
    df["DIQ070"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


diq070_mismatch = int(
    np.sum(
        diq070_expected
        !=
        diq070_present
    )
)


if int(
    diq070_expected.sum()
) != 1674:

    raise ValueError(
        "Unexpected DIQ070 eligibility count."
    )


if diq070_mismatch != 0:

    raise ValueError(
        "DIQ070 eligibility/routing differs."
    )


print(
    "PASS: DIQ070 routing validated."
)


# ============================================================
# 28. DIABETES CARE QUESTIONS
# ============================================================

check_frequency(
    "DIQ230",
    {
        1: 235,
        2: 73,
        3: 89,
        4: 146,
        5: 339,
        9: 10
    },
    8005
)


check_frequency(
    "DIQ240",
    {
        1: 694,
        2: 197,
        9: 1
    },
    8005
)


# ============================================================
# 29. DID250 - DOCTOR VISITS
# ============================================================

check_range_special(
    "DID250",
    1,
    40,
    682,
    {
        0: 9,
        7777: 0,
        9999: 3
    },
    8203
)


did250_expected = (
    df["DIQ240"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


did250_present = (
    df["DID250"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


did250_routing_mismatch = int(
    np.sum(
        did250_expected
        !=
        did250_present
    )
)


if did250_routing_mismatch != 0:

    raise ValueError(
        "DIQ240/DID250 routing differs."
    )


print(
    "PASS: DIQ240/DID250 routing validated."
)


# ============================================================
# 30. DID260 - BLOOD GLUCOSE CHECKS
# ============================================================

check_range_special(
    "DID260",
    1,
    50,
    713,
    {
        0: 178,
        777: 0,
        999: 2
    },
    8004
)


# ============================================================
# 31. DIQ260U
# ============================================================

check_frequency(
    "DIQ260U",
    {
        1: 430,
        2: 176,
        3: 78,
        4: 29
    },
    8184
)


diq260u_expected = (
    df["DID260"]
    .between(
        1,
        50
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq260u_present = (
    df["DIQ260U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


diq260u_mismatch = int(
    np.sum(
        diq260u_expected
        !=
        diq260u_present
    )
)


if diq260u_mismatch != 0:

    raise ValueError(
        "DID260/DIQ260U routing differs."
    )


print(
    "PASS: DID260/DIQ260U routing validated."
)


# ============================================================
# 32. A1C QUESTIONS
# ============================================================

check_frequency(
    "DIQ275",
    {
        1: 749,
        2: 107,
        9: 36
    },
    8005
)


check_frequency(
    "DIQ291",
    {
        1: 250,
        2: 225,
        3: 40,
        4: 2,
        5: 11,
        6: 122,
        77: 2,
        99: 97
    },
    8148
)


# ============================================================
# 33. DIQ275 -> DIQ280 / DIQ291 ROUTING
# ============================================================

diq275_yes = (
    df["DIQ275"]
    .eq(1)
    .fillna(False)
)


#
# DIQ280:
#
# 749 eligible
# 747 populated
# 2 legitimately missing
# no DIQ280 values outside eligible rows
#

diq280_outside = int(
    (
        df["DIQ280"].notna()
        &
        ~diq275_yes
    )
    .sum()
)


diq280_eligible_missing = int(
    (
        diq275_yes
        &
        df["DIQ280"].isna()
    )
    .sum()
)


if diq280_outside != 0:

    raise ValueError(
        "DIQ280 populated outside DIQ275=1."
    )


if diq280_eligible_missing != 2:

    raise ValueError(
        "Unexpected DIQ280 eligible-missing count."
    )


print(
    "PASS: DIQ280 routing validated "
    "(2 eligible records are missing)."
)


#
# DIQ291 is populated for all 749 DIQ275=1 records.
#

diq291_expected = (
    diq275_yes
    .to_numpy(
        dtype=bool
    )
)


diq291_present = (
    df["DIQ291"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        diq291_expected
        !=
        diq291_present
    )
) != 0:

    raise ValueError(
        "DIQ275/DIQ291 routing differs."
    )


print(
    "PASS: DIQ275/DIQ291 routing validated."
)


# ============================================================
# 34. BLOOD PRESSURE / LDL SELF-REPORT RANGES
# ============================================================

check_range_special(
    "DIQ300S",
    58,
    212,
    604,
    {
        7777: 4,
        9999: 276
    },
    8013
)


check_range_special(
    "DIQ300D",
    30,
    150,
    589,
    {
        7777: 2,
        9999: 293
    },
    8013
)


check_range_special(
    "DID310S",
    72,
    180,
    393,
    {
        6666: 305,
        7777: 2,
        9999: 186
    },
    8011
)


check_range_special(
    "DID310D",
    30,
    140,
    373,
    {
        6666: 305,
        7777: 3,
        9999: 205
    },
    8011
)


check_range_special(
    "DID320",
    8,
    340,
    146,
    {
        5555: 24,
        6666: 31,
        7777: 12,
        9999: 672
    },
    8012
)


check_range_special(
    "DID330",
    15,
    250,
    100,
    {
        6666: 284,
        7777: 7,
        9999: 440
    },
    8066
)


# ============================================================
# 35. DID341 - DOCTOR FOOT CHECKS
# ============================================================

check_range_special(
    "DID341",
    1,
    106,
    631,
    {
        0: 244,
        7777: 0,
        9999: 7
    },
    8015
)


# ============================================================
# 36. DID350 - PERSONAL FOOT CHECKS
# ============================================================

check_range_special(
    "DID350",
    1,
    30,
    712,
    {
        0: 168,
        7777: 0,
        9999: 2
    },
    8015
)


# ============================================================
# 37. DIQ350U ROUTING
# ============================================================

check_frequency(
    "DIQ350U",
    {
        1: 435,
        2: 179,
        3: 73,
        4: 25
    },
    8185
)


diq350u_expected = (
    df["DID350"]
    .between(
        1,
        30
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq350u_present = (
    df["DIQ350U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


diq350u_mismatch = int(
    np.sum(
        diq350u_expected
        !=
        diq350u_present
    )
)


if diq350u_mismatch != 0:

    raise ValueError(
        "DID350/DIQ350U routing differs."
    )


print(
    "PASS: DID350/DIQ350U routing validated."
)


# ============================================================
# 38. EYE EXAM / RETINOPATHY
# ============================================================

check_frequency(
    "DIQ360",
    {
        1: 89,
        2: 490,
        3: 125,
        4: 112,
        5: 59,
        9: 10
    },
    8012
)


check_frequency(
    "DIQ080",
    {
        1: 175,
        2: 702,
        9: 8
    },
    8012
)


# ============================================================
# 39. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original XPT contained ordinary zero count = 0.
#
# Therefore after repair the ONLY zeros must be:
#
#     DID250 =   9
#     DID260 = 178
#     DID341 = 244
#     DID350 = 168
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 599:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the 599 validated restorations."
)


# ============================================================
# 40. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Only validated tiny -> zero changes are permitted.
#
# DIQ280 remains completely untouched.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 54 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 41. STORE MISSING COUNTS
# ============================================================

expected_missing_counts = {
    col: int(
        expected_df[col]
        .isna()
        .sum()
    )
    for col in EXPECTED_COLUMNS
}


# ============================================================
# 42. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# DO NOT use:
#
#     df.round(...)
#
# DO NOT use:
#
#     float_format=
#
#
# This is especially important for DIQ280.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 43. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 44. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 54 DIQ_J variables."
)


# ============================================================
# 45. MISSING-VALUE FIDELITY
# ============================================================

missing_position_mismatches = 0

missing_count_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    actual_missing_count = int(
        roundtrip_df[col]
        .isna()
        .sum()
    )


    if (
        actual_missing_count
        !=
        expected_missing_counts[col]
    ):

        missing_count_failures.append(
            col
        )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


if missing_count_failures:

    raise ValueError(
        "Missing counts changed in:\n"
        + str(
            missing_count_failures
        )
    )


print(
    "PASS: Missing positions/counts preserved exactly."
)


# ============================================================
# 46. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 47. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 48. RAW CSV .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# CRITICAL:
#
# DO NOT apply this check globally.
#
# DIQ280 is a real continuous A1C field.
#
# Therefore ".0" is perfectly legitimate in DIQ280.
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in integer fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in the 53 whole-number fields."
)


# ============================================================
# 49. DIQ280 RAW DECIMAL PRESERVATION CHECK
# ============================================================

#
# DO NOT treat ".0" as an error in DIQ280.
#
# Examples can legitimately remain:
#
#     5.0
#     6.0
#     7.0
#
# alongside:
#
#     5.7
#     6.4
#     7.2
#

diq280_dot_zero_count = int(
    (
        raw_csv_df["DIQ280"]
        .ne("")
        &
        raw_csv_df["DIQ280"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- DIQ280 RAW DECIMAL CHECK ---"
)

print(
    "DIQ280 '.0' tokens:",
    diq280_dot_zero_count
)

print(
    "These are legitimate continuous-field values."
)


#
# Test export of attached file produces 405
# whole-looking float tokens.
#

if diq280_dot_zero_count != 405:

    raise ValueError(
        "Unexpected DIQ280 whole-looking float "
        "token count."
    )


print(
    "PASS: DIQ280 decimal-column formatting preserved."
)


# ============================================================
# 50. EXACT DIQ280 NUMERIC ROUND-TRIP
# ============================================================

expected_diq280 = (
    expected_df["DIQ280"]
    .to_numpy(
        dtype=float
    )
)


exported_diq280 = (
    pd.to_numeric(
        raw_csv_df["DIQ280"]
        .where(
            raw_csv_df["DIQ280"] != "",
            np.nan
        ),
        errors="coerce"
    )
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_diq280,
    exported_diq280,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "DIQ280 numerical values changed "
        "during CSV export."
    )


print(
    "PASS: DIQ280 values preserved exactly."
)


# ============================================================
# 51. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 599 validated zeros."
)


# ============================================================
# 52. FINAL ROUTING CHECKS
# ============================================================

print(
    "\n--- FINAL ROUTING CHECKS ---"
)


routing_results = {
    "DIQ172_DIQ175": risk_routing_mismatch,
    "DIQ050_DID060": did060_mismatch,
    "DIQ070": diq070_mismatch,
    "DIQ240_DID250": did250_routing_mismatch,
    "DID260_DIQ260U": diq260u_mismatch,
    "DID350_DIQ350U": diq350u_mismatch
}


print(
    routing_results
)


if any(
    value != 0
    for value in routing_results.values()
):

    raise ValueError(
        "One or more routing checks failed."
    )


print(
    "PASS: All tested routing relationships "
    "have zero mismatches."
)


# ============================================================
# 53. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 54. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 55. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DIQ_J CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2017-2018"
)


print(
    "Component: Questionnaire - Diabetes"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "DID250 restored zeros:",
    int(
        df["DID250"]
        .eq(0)
        .sum()
    )
)


print(
    "DID260 restored zeros:",
    int(
        df["DID260"]
        .eq(0)
        .sum()
    )
)


print(
    "DID341 restored zeros:",
    int(
        df["DID341"]
        .eq(0)
        .sum()
    )
)


print(
    "DID350 restored zeros:",
    int(
        df["DID350"]
        .eq(0)
        .sum()
    )
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "DIQ280 genuine fractional observations:",
    fractional_counts["DIQ280"]
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "Integer-field '.0' errors:",
    "0 expected"
)


print(
    "DIQ280 '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "Only the 599 validated XPORT tiny-zero "
    "artifacts are changed to real numeric zero."
)


print(
    "DIQ280 is preserved as a genuine "
    "continuous A1C decimal field."
)


print(
    "Only the other 53 mathematically whole-number "
    "fields are converted to nullable Int64."
)


print(
    "No df.round() is used to modify source data."
)


print(
    "No float_format is used during CSV export."
)


print(
    "Genuine Dietary and all other genuine "
    "continuous decimal values remain exactly AS IS."
)

NHANES 2017-2018 DIQ_J XPT -> CSV
Rows: 8,897
Columns: 54
PASS: Structure = 8,897 rows x 54 columns.

--- FIELD TYPE PLAN ---
Character fields: []
Genuine decimal fields: ['DIQ280']
Whole-number/code fields: 53

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 599
Variables containing tiny artifacts: 4

Tiny values by variable:
DID341    244
DID260    178
DID350    168
DID250      9

PASS: Exact 599-value tiny-zero pattern across 4 variables validated.

--- VALIDATED ZERO RESTORATION ---
DID250: 9 restored to 0
DID260: 178 restored to 0
DID341: 244 restored to 0
DID350: 168 restored to 0

PASS: 599 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE DETECTION ---
Variables with fractional observations: ['DIQ280']
PASS: DIQ280 is the only genuine decimal variable.
DIQ280 fractional observations: 342

--- DIQ280 A1C FIDELITY ---
Valid 2.3-14: 490
Fractional valid A1C: 342
Whole-lo